In [3]:
import pandas as pd
import pymysql
from sqlalchemy import inspect, text
from core.database_utils import get_db_engine, get_database_inventory

In [2]:
# Solo necesitas llamar a la función
engine = get_db_engine()

if engine:
    # Ahora puedes obtener el inventario completo automáticamente
    inventory = get_database_inventory()

--- Connection to 52.249.30.165 established successfully ---
--- Connection to 52.249.30.165 established successfully ---
--- Database Inventory: granerosguerra_copiaenrique ---
Total tables found: 0



In [11]:


def run_query(sql_query, description="Data"):
    """
    Executes a SQL query and returns a pandas DataFrame.
    :param sql_query: The SQL text or SQLAlchemy text() object.
    :param description: A label for printing status messages.
    :return: Pandas DataFrame or None if failed.
    """
    try:
        # 1. Ensure the query is wrapped in text() if it's a raw string
        if isinstance(sql_query, str):
            sql_query = text(sql_query)
            
        with engine.connect() as conn:
            df = pd.read_sql_query(sql_query, conn)
            
        # 2. Check and report status
        if df.empty:
            print(f"--- Warning: No records found for [{description}]. ---")
            return df
        else:
            print(f"--- Success: [{description}] retrieved with {len(df)} rows. ---")
            return df
            
    except Exception as e:
        print(f"--- Critical Error executing [{description}]: {e} ---")
        return None



--- Success: [Table Inventory] retrieved with 45 rows. ---


In [9]:

# Definir la consulta para listar tablas del esquema 'dbo' o todos
query_tables = text("""
    SELECT 
    s.name AS [Schema],
    t.name AS [Table],
    c.name AS [Column],
    pc.name AS [DataType],
    c.max_length AS [MaxLength],
    c.is_nullable AS [IsNullable]
FROM sys.columns c
JOIN sys.tables t ON c.object_id = t.object_id
JOIN sys.schemas s ON t.schema_id = s.schema_id
JOIN sys.types pc ON c.user_type_id = pc.user_type_id
WHERE t.name IN ('CATALOGO_CLIENTES_DATA', 'COBRANZA_DATA', 'CREDITO_DATA', 'DESTINOS_CLIENTES')
ORDER BY t.name, c.column_id;
""")

views_tables = text("""
SELECT 
    s.name AS [Schema],
    v.name AS [View_Name],
    c.name AS [Column_Name],
    pc.name AS [DataType]
FROM sys.columns c
JOIN sys.views v ON c.object_id = v.object_id
JOIN sys.schemas s ON v.schema_id = s.schema_id
JOIN sys.types pc ON c.user_type_id = pc.user_type_id
ORDER BY v.name, c.column_id;
""")

In [16]:
df_tab = run_query(query_tables, "Table inventory")
print(df_tab)


--- Success: [Table inventory] retrieved with 45 rows. ---
     Schema                   Table             Column  DataType  MaxLength  \
0   proadel  CATALOGO_CLIENTES_DATA             CODIGO  nvarchar         40   
1   proadel  CATALOGO_CLIENTES_DATA             NOMBRE  nvarchar        300   
2   proadel  CATALOGO_CLIENTES_DATA          DIRECCION  nvarchar        300   
3   proadel  CATALOGO_CLIENTES_DATA           TELEFONO  nvarchar        100   
4   proadel  CATALOGO_CLIENTES_DATA  LIMITE_DE_CREDITO   decimal          9   
5   proadel  CATALOGO_CLIENTES_DATA      TOTAL_CREDITO   decimal          9   
6   proadel  CATALOGO_CLIENTES_DATA     TOTAL_COBRANZA   decimal          9   
7   proadel  CATALOGO_CLIENTES_DATA    TOTAL_DEV_VENTA   decimal          9   
8   proadel  CATALOGO_CLIENTES_DATA      SALDO_INICIAL   decimal          9   
9   proadel  CATALOGO_CLIENTES_DATA             ESTADO  nvarchar        100   
10  proadel  CATALOGO_CLIENTES_DATA   TOTAL_POR_ABONAR   decimal        